<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex05-cnn-and-gnn/Ex05_00_environment_check.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Prince, *Understanding Deep Learning*, MIT Press 2023.
- Goodfellow, Bengio & Courville, *Deep Learning*, MIT Press 2016.  Ch. 10 for recurrence and the LSTM.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_05 · Notebook 00 — Environment Check

**Deep Learning for Engineering · Aalborg University · Part 1 · paired with L5.1 and L5.2**

## What Ex_05 is about

One idea, three kinds of data: **a learned local rule, applied everywhere.** A
convolution applies it across the pixels of an image, message passing applies it
across the buses of a network, and a recurrent network applies it along time.
Lecture block L5 introduced the three. In Ex_05 you build each one, train it on
an engineering problem, and compare it with a simpler model on the same data.

### Goals

By the end of the exercise set you can

1. build a small convolutional network (CNN), set a convolution's weights by
   hand, and show that the trained CNN beats a dense network on the same images
   with far fewer parameters;
2. build an adjacency matrix and perform message passing by hand;
3. show permutation equivariance to machine precision, and a dense network
   failing the same test;
4. train a graph network to approximate the AC power flow on six buses, compare
   it with a dense network, and weigh its accuracy against its speed — also after
   a line trips;
5. build a recurrent network and an LSTM, and compare them with the persistence
   forecast on the same held-out data.

### Method — six notebooks, run in order

| notebook | what you build | data | lecture |
|---|---|---|---|
| **00** | nothing — checks that the tools and the data work | small samples of all three datasets | — |
| **01** | a CNN, against a dense network | 600 weld radiographs | L5.1 |
| **02** | adjacency, neighbours, message passing, in NumPy | the six-bus network | L5.1 |
| **03** | a graph network that predicts the power flow | 800 AC power-flow cases on six buses | L5.1 |
| **04** | a recurrent network and an LSTM, against persistence | 40 days of hourly substation load | L5.2 |
| **05** | the report | the results saved by 01, 03 and 04 | — |

Notebooks 01, 03 and 04 each save their results to `Ex05_outputs/`, and notebook 05
builds the report from them.

### Applications

* **Inspection.** Sort 16 × 16 weld radiographs into clean, cracked and pitted. The
  defect can sit anywhere on the plate, which is why one detector applied
  everywhere is the right tool.
* **Power networks.** From what is given at each bus — P and Q at a load, P and
  |V| at a generator — predict the voltage magnitude and angle at every bus. The
  targets come from a full AC power flow, the ground truth the graph network
  learns to approximate faster; the line impedances are invented, and notebook 03
  says which parts are real.
* **Forecasting.** Predict the next hour of substation demand from the previous
  twenty-four.

## What this notebook checks

Run it first, top to bottom. There is nothing to write in it: every cell is
complete. Its job is to establish, before you spend an hour on anything else, that

* `torch`, `numpy` and `matplotlib` are importable and recent enough;
* `Ex_5_core.py` is on the path and imports cleanly;
* the three datasets can be generated on **your** machine with no network
  connection — the datasets here are small samples of the ones the numbered
  notebooks train on;
* a convolution, a graph step and a recurrent step all run.

**Nothing is downloaded.** That is a deliberate choice rather than an oversight.
The usual first exercise in a convolutional-network course fetches MNIST from a
mirror, and every year that mirror is slow, blocked by a university firewall, or
has moved. The three image classes here are drawn procedurally in about twenty
lines of NumPy, and they are the same three classes on every machine in the
room.

If a cell in this notebook fails, fix it before going on. A missing package is a
two-minute problem now and a lost afternoon in notebook 03.

---

## 0 · Versions

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['Ex_5_core.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex05-cnn-and-gnn/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
import sys
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import torch

print("python     ", sys.version.split()[0])
print("numpy      ", np.__version__)
print("matplotlib ", matplotlib.__version__)
print("torch      ", torch.__version__)
print("cuda        available:", torch.cuda.is_available(), " (not needed)")

**What you should see.** Four version numbers and `cuda available: False`
on most machines.

Any Python from 3.9 and any PyTorch from 2.0 will do. **No GPU is required
anywhere in Ex_05.** The largest model in this exercise set has under twenty
thousand parameters; a GPU would spend more time moving the data than computing
on it.

---

## 1 · The shared module

In [ ]:
import Ex_5_core as core

print("module file:", core.__file__)
print("output dir :", core.OUTPUT_DIR)
print()
print("classes    :", core.CLASS_NAMES)
print("buses      :", core.BUS_NAMES)
print("bus types  :", core.BUS_TYPE)
print("lines      :", len(core.SIX_BUS_LINES))

**What you should see.** The path to `Ex_5_core.py`, a path ending in
`Ex05_outputs`, the three image classes `('clean', 'crack', 'pit')`, the six buses
`('gen (slack)', 'gen (PV)', 'load', 'load', 'load', 'HVDC')`, their power-flow
types `('slack', 'PV', 'PQ', 'PQ', 'PQ', 'PQ')`, and `lines : 8`.

If the import fails with `ModuleNotFoundError`, the notebook is not being run
from the `Ex05-cnn-and-gnn` folder. On Colab, upload the whole folder and
`%cd` into it.

---

## 2 · Dataset one — the weld radiographs

Sixty images, twenty of each class, 16 by 16 pixels, generated here and now.
This is a sample to check the generator: notebook 01 trains on six hundred,
drawn the same way.

In [ ]:
import Ex_5_core as core

X_img, y_img = core.weld_images(n_per_class=20, seed=3)

print("images :", X_img.shape, X_img.dtype)
print("labels :", y_img.shape, "counts", np.bincount(y_img))
print("pixels : min %.3f  max %.3f" % (X_img.min(), X_img.max()))

core.plot_images(X_img, y_img, n=12, title="Twelve synthetic weld radiographs")
plt.show()

**What you should see.** `images : (60, 1, 16, 16) float32`, label counts
`[20 20 20]`, pixel values inside [0, 1], and a grid of twelve small grey
squares: some plain, some crossed by a thin dark line, some with a small dark
dot.

The shape `(60, 1, 16, 16)` is `(batch, channels, height, width)`. The `1` is
the single greyscale channel. `torch.nn.Conv2d` insists on that four-dimensional
layout, and forgetting the channel axis is the most common first error with a
convolutional network — you will meet it in notebook 01 whether you want to or
not.

Look at the pictures for a moment. The defect is in a different place in every
image. That is the whole reason a convolution is the right tool: the same
detector has to work everywhere on the plate.

---

## 3 · Dataset two — the six-bus network

The object notebooks 02 and 03 are built around: six buses — two generators,
three load centres and an HVDC infeed — joined by eight lines. Only the network
itself is loaded here; notebook 03 solves the power-flow cases it trains on.

In [ ]:
import Ex_5_core as core

A = core.six_bus_adjacency()

print("adjacency (6 x 6):")
print(A.astype(int))
print()
print("degrees:", A.sum(axis=1).astype(int))
print("symmetric:", np.allclose(A, A.T))
print()
print("the eight lines, per unit on 100 MVA:")
print(core.line_table())

core.plot_graph(title="The six-bus network")
plt.show()

**What you should see.** A symmetric matrix of zeros and ones with a zero
diagonal, degrees `[2 3 3 3 3 2]`, `symmetric: True`, a table of the eight lines
with their resistance R, reactance X and admittance y = 1/(R + jX), and a picture
of six circles joined by eight lines.

**The lines.** A line is described by its series **impedance** $z = R + jX$, or
equivalently by its **admittance** $y = 1/z$. X is about eight times R, which is
typical of transmission lines. The numbers are invented.

**Symmetry.** A transmission line carries power in both directions, so the graph
is undirected. Not every engineering graph is: a pipe network with check valves
is not, and a road network with one-way streets is not.

**Bus types.** A power flow has four quantities at each bus — P, Q, |V| and θ —
and two of them are given. Bus 0 is the **slack** bus: its |V| and its angle,
θ = 0, are given, and it supplies whatever power the others do not, including
the losses in the lines. Bus 1 is a **PV** bus: a generator with P and |V| given.
Buses 2 to 5 are **PQ** buses, with P and Q given. Notebook 03 builds on this.

---

## 4 · Dataset three — the load profile

Ten days of hourly substation demand, cut into 24-hour windows. Notebook 04 uses
forty days of the same series.

In [ ]:
import Ex_5_core as core

series = core.load_profile(n_days=10, seed=21)

print("series :", series.shape, "  min %.3f  max %.3f  mean %.3f"
      % (series.min(), series.max(), series.mean()))

X_seq, y_seq = core.make_windows(series, window=24)
print("windows:", X_seq.shape, "targets:", y_seq.shape)

core.plot_series(series, title="Ten days of synthetic substation demand",
                 highlight=(48, 72))
plt.show()

**What you should see.** `series : (240,)` with values between about 0.19
and 0.80, `windows: (216, 24, 1)` and `targets: (216, 1)`, and a wiggly trace
with a clear daily rhythm — a morning shoulder and a taller evening peak — and
two lighter days at the weekend.

`(216, 24, 1)` is `(batch, time, features)`, which is what every recurrent layer
in PyTorch expects when you pass `batch_first=True`. One feature, because this
is a single measured channel.

---

## 5 · One convolution, one graph step, one recurrent step

Three lines of arithmetic, one from each half of lecture block L5. If these
three cells run, everything in Ex_05 will run.

In [ ]:
import Ex_5_core as core

import torch.nn as nn

core.set_seed(0)

# (a) a convolution: one 3x3 kernel over one 16x16 image
conv = nn.Conv2d(in_channels=1, out_channels=4, kernel_size=3, padding=1)
out_conv = conv(torch.tensor(X_img[:2]))
print("(a) conv    ", tuple(torch.tensor(X_img[:2]).shape), "->",
      tuple(out_conv.shape), " parameters:", core.count_parameters(conv))

# (b) one round of message passing: every bus adds up its neighbours' features
H = torch.eye(6)                                        # bus i starts as a one-hot vector: "I am bus i"
out_graph = torch.tensor(A, dtype=torch.float32) @ H    # row i: the sum over bus i's neighbours
print("(b) graph   ", tuple(H.shape), "->", tuple(out_graph.shape),
      " bus 0 heard from buses:", torch.nonzero(out_graph[0]).ravel().tolist())

# (c) a recurrent step: fold one time step into a hidden state
cell = nn.Linear(1 + 8, 8)
h = torch.zeros(5, 8)
x_t = torch.tensor(X_seq[:5, 0, :])
h = torch.tanh(cell(torch.cat([x_t, h], dim=1)))
print("(c) recurrent hidden state ->", tuple(h.shape),
      " parameters:", core.count_parameters(cell))

**What you should see.**

```
(a) conv     (2, 1, 16, 16) -> (2, 4, 16, 16)  parameters: 40
(b) graph    (6, 6) -> (6, 6)  bus 0 heard from buses: [1, 2]
(c) recurrent hidden state -> (5, 8)  parameters: 80
```

Three things worth noticing before you close this notebook.

**Forty parameters.** The convolution turned a 16 by 16 image into four 16 by 16
feature maps — 1024 numbers out of 256 — using forty parameters: four kernels of
nine weights, plus four biases. A dense layer doing the same thing would need
$256 \times 1024 = 262{,}144$ weights. That ratio is the argument of L5.1's
first half, and notebook 01 counts it again for its own first layer: 80 against
526,336.

**One round reaches one hop.** Row $i$ of $AH$ is the sum of bus $i$'s
neighbours' rows. Each bus started as a vector saying only "I am bus $i$", so
after one round bus 0 has heard from buses 1 and 2, its neighbours, and from
nobody else. That is message passing: gather from the neighbours, add up, and —
in a real layer — combine the sum with the bus's own state. Notebook 02 builds it
by hand, and notebook 03 stacks three rounds, one per hop across the network.

**The recurrent cell has no time index in it.** It is one `Linear` layer, applied
once per time step, with its own previous output fed back in. That single reused
layer *is* the recurrent network — the "unrolling" you saw in L5.2 is a picture
of a `for` loop, not of extra parameters.

---

## 6 · Ready — and which version to continue with

If every cell above ran, you are set up. Every numbered notebook in this set exists in **two versions with the same
text, the same figures and the same questions**. They differ only in whether
the code is already written.

| | continue with | the code |
|---|---|---|
| **write it yourself** | [`Ex05_01_cnn_image_classification.ipynb`](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex05-cnn-and-gnn/Ex05_01_cnn_image_classification.ipynb) | cells marked `TODO` are yours to write |
| **read and run** | [`Ex05_01_cnn_image_classification_light.ipynb`](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex05-cnn-and-gnn/Ex05_01_cnn_image_classification_light.ipynb) | complete: every cell runs as it stands |

Pick one and stay on it: each notebook links on to the next of the same kind,
and both tracks end at the same `Ex05_05_report.ipynb`. Switching is fine too - if a TODO
defeats you, open the same notebook's complete version, read that cell, and
carry on where you were. **The report is the same either way, and it is what
is marked**, so the complete version is not a shortcut past the work: it moves
the work from typing the code to reading it and explaining what it did.

A note on time. Every notebook runs on a laptop CPU. Notebook 03 is the
longest, because it trains seven networks, five of them in its depth sweep,
and times the power flow on grids of up to 768 buses; allow a few minutes for it
on Colab. The others need well under a minute of
computing each.